In [0]:
spark.conf.set("spark.sql.shuffle.partitions", "auto")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

In [0]:
%pip install -q xgboost scikit-learn pandas numpy pyarrow mosaicml-streaming
dbutils.library.restartPython()

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
import numpy as np
import os
import pandas as pd
import xgboost as xgb
from sklearn.metrics import ndcg_score, roc_auc_score
from streaming import StreamingDataset
import mlflow
from tqdm import tqdm
import shap
import matplotlib.pyplot as plt
from pyspark.sql import DataFrame
from sklearn.preprocessing import LabelEncoder
import seaborn as sns
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
import optuna
import joblib
import config

In [0]:
# Widen the hackathon output from the top 25 to the top 100 ranked themes per customer.
base = spark.sql('select * from ds_sandbox.next_uk_nextAds_predict_prod_ranked where simple_rules_rank <=100')

base.cache().count()

In [0]:
features = config.features
model_name = "marketingdata_prod.ds_sandbox.nextads_hackathon_model"
model_version = "1" # Increments every time you re-run your save block
model_uri = f"models:/{model_name}/{model_version}"

predict_input = base.select(
    "account_number",
    "theme_clean",
    "baskets_behavior__recency_rank",
    *features,
)

inference_partitions = max(base.rdd.getNumPartitions(), spark.sparkContext.defaultParallelism)
predict_input = predict_input.repartition(inference_partitions, "account_number")

print(f"Loading model: {model_uri}")
print(f"Predict input partitions: {predict_input.rdd.getNumPartitions()}")

In [0]:
encoders = joblib.load("ranking_encoders.joblib")
encoders_bc = spark.sparkContext.broadcast(encoders)

prediction_schema = (
    predict_input.select(
        F.col("account_number"),
        F.col("theme_clean").alias("theme"),
        F.col("month"),
        F.col("baskets_behavior__recency_rank"),
        F.lit(0.0).cast("float").alias("prediction"),
    )
    .schema
)

In [0]:
def predict_partition(iterator):
    import mlflow
    import numpy as np
    import xgboost as xgb

    partition_encoders = encoders_bc.value

    global _hackathon_predict_model
    try:
        model = _hackathon_predict_model
    except NameError:
        _hackathon_predict_model = mlflow.xgboost.load_model(model_uri)
        model = _hackathon_predict_model

    for pdf in iterator:
        if pdf.empty:
            continue

        feature_pdf = pdf[features].copy()

        for col, encoder in partition_encoders.items():
            if col not in feature_pdf.columns:
                continue

            values = pdf[col].astype(str)
            valid_mask = values.isin(encoder.classes_)
            safe_values = np.where(valid_mask, values, encoder.classes_[0])
            encoded_values = encoder.transform(safe_values).astype(np.int32, copy=False)
            encoded_values[~valid_mask.to_numpy()] = -1
            feature_pdf[col] = encoded_values

        X = feature_pdf.to_numpy(dtype=np.float32, copy=False)
        dmatrix = xgb.QuantileDMatrix(X, feature_names=features)
        preds = model.predict(dmatrix).astype(np.float32, copy=False)

        result = pdf[["account_number", "theme_clean", "month", "baskets_behavior__recency_rank"]].copy()
        result = result.rename(columns={"theme_clean": "theme"})
        result["prediction"] = preds
        yield result

In [0]:
predictions = predict_input.mapInPandas(predict_partition, schema=prediction_schema)
predictions.cache().count()

In [0]:
display(predictions.limit(20))

In [0]:
output_table = "ds_sandbox.next_uk_nextAds_predict_prod_half"
output_table

In [0]:
predictions.write.mode("overwrite").saveAsTable(output_table)

In [0]:
spark.table(output_table).count()

In [0]:
display(spark.table(output_table).limit(20))